In [ ]:
# Imports
import random
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import keras

In [ ]:
# Change pixel values to be in range [0,1]
def normalize_img(image, label):
	"""Normalizes images: `uint8` -> `float32`."""
	return tf.cast(image, tf.float32) / 255., label

In [ ]:
# Load data
ds, ds_test = tfds.load('cifar10', split=['train[:4000]', 'test[:1000]'], shuffle_files=False, as_supervised=True)
ds = ds.map(normalize_img, num_parallel_calls=tf.data.AUTOTUNE)
ds_test = ds_test.map(normalize_img, num_parallel_calls=tf.data.AUTOTUNE)

In [ ]:
# Randomly sample queries from the dataset to train the target model

zeros = np.zeros(int(len(ds)/2))
ones = np.ones(int(len(ds)/2))
				 

filter = np.concatenate((zeros, ones))
random.shuffle(filter)

filtered_features = []
filtered_labels = []

for step, (x, y) in enumerate(ds):
	if filter[step] == 1:
		filtered_features.append(x)
		filtered_labels.append(y)

filtered_ds = tf.data.Dataset.from_tensor_slices((filtered_features, filtered_labels))

In [ ]:
# Generate N subsets of data
# Each query from the original dataset will appear in N/2 subsets
# Each subset will be used to train an imitative model

num_subsets = 10

subsets = [[] for i in range(num_subsets)]
features = [[] for i in range(num_subsets)]
labels = [[] for i in range(num_subsets)]

for step, (x, y) in enumerate(ds):
	for offset in range(int(num_subsets / 2)):
		index = (step + offset) % num_subsets
		features[index].append(x)
		labels[index].append(y)

subsets = [tf.data.Dataset.from_tensor_slices((features[i], labels[i])) for i in range(num_subsets)]

In [ ]:
# Cache and prefetch datasets for faster performance

ds = ds.cache()
ds = ds.batch(1)
ds = ds.prefetch(tf.data.AUTOTUNE)

ds_test = ds_test.cache()
ds_test = ds_test.batch(64)
ds_test = ds_test.prefetch(tf.data.AUTOTUNE)

filtered_ds = filtered_ds.cache()
filtered_ds = filtered_ds.batch(64)
filtered_ds = filtered_ds.prefetch(tf.data.AUTOTUNE)


for i in range(len(subsets)):
	subsets[i] = subsets[i].cache()
	subsets[i] = subsets[i].batch(64)
	subsets[i] = subsets[i].prefetch(tf.data.AUTOTUNE)

In [ ]:
# Architecture used by target model and imitative models

def get_classifier():
	model = tf.keras.models.Sequential([
	tf.keras.layers.Conv2D(128, (3, 3), activation="relu", input_shape=(32, 32, 3)),
	tf.keras.layers.MaxPool2D(2,2),
	tf.keras.layers.Conv2D(128, (3, 3), activation="relu"),
	tf.keras.layers.MaxPool2D(2,2),
	tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
	tf.keras.layers.MaxPool2D(2,2),
	tf.keras.layers.Flatten(),
	tf.keras.layers.Dense(64, activation='relu'),
	tf.keras.layers.Dense(10)
	])

	return model

In [ ]:
# Train target model on in data

target_model = get_classifier()

target_model.compile(
optimizer=tf.keras.optimizers.Adam(0.001),
loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
metrics=[tf.keras.metrics.SparseCategoricalAccuracy()],
)

target_model.fit(
    filtered_ds,
    epochs=50
)

In [ ]:
# Train and save imitative models

mse = keras.losses.MeanSquaredError()
ce = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# Controls influence of imitative loss
alpha = 0.5

# Controls confidence of the logits
temperature = 2

num_classes = target_model.output_shape[-1]

# Each model is trained on a different subset of data
for i in range(num_subsets):
	optimizer = tf.keras.optimizers.Adam(0.001)
	im_model = get_classifier()

	warmup = True

	if warmup:
		# Warmup model to achieve reasonable accuracy
		for epoch in range(25):

			subsets[i] = subsets[i].shuffle(buffer_size=10)

			for step, (x_batch, y_batch) in enumerate(subsets[i]):
				with tf.GradientTape() as tape:
					student_logits = im_model(x_batch, training=False)

					loss_value = ce(y_batch, student_logits)
					
				grads = tape.gradient(loss_value, im_model.trainable_weights)

				optimizer.apply_gradients(zip(grads, im_model.trainable_weights))


	# Next, train model with mixture of regular and imitative loss
	best_loss = 999
	for epoch in range(40):

		subsets[i] = subsets[i].shuffle(buffer_size=10)

		for step, (x_batch, y_batch) in enumerate(subsets[i]):
			with tf.GradientTape() as tape:
				student_logits = im_model(x_batch, training=True) / temperature

				teacher_logits = target_model(x_batch) / temperature

				# Certain output logits are weighted higher as they are related to membership
				weight_matrix = [[1 for i in range(num_classes)] for j in range(len(x_batch))]
				for query_index in range(len(x_batch)):
					copy = tf.identity(teacher_logits[query_index]).numpy()
					copy[y_batch[query_index]] = float('-inf')
					max_index = np.argmax(copy)

					# Output logit of correct class is weighted higher
					weight_matrix[query_index][y_batch[query_index]] = np.sqrt(num_classes)

					# Output logit of most confident prediction that's not the correct class is weighted higher
					weight_matrix[query_index][max_index] = np.sqrt(num_classes)

				weighted_student_logits = weight_matrix * student_logits
				weighted_teacher_logits = weight_matrix * teacher_logits

				# Loss measuring how closely this imitative model follows the target model
				imitate_loss = mse(weighted_teacher_logits, weighted_student_logits)

				# Normalize imitative loss to scale of regular loss
				avg_weight = np.mean(weight_matrix)
				imitate_loss = imitate_loss / avg_weight

				# Loss measuring accuracy of this imitative model
				regular_loss = ce(y_batch, student_logits)

				# Both losses are combined to train the model
				loss_value = alpha * imitate_loss + (1 - alpha) * regular_loss

				# Evaluate model on test data
				# Model should have good test accuracy
				test_loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
				batch_loss = 0
				for test_step, (x_test, y_test) in enumerate(ds_test):
					pred = im_model.predict(x_test, verbose=0)

					batch_loss += test_loss_fn(y_test, pred)
				test_loss = batch_loss/len(ds_test)

				# Save best model
				if (test_loss < best_loss):
					best_loss = test_loss
					im_model.save_weights(f'./imitative_models/im_model_{i+1}.weights.h5')
				
			grads = tape.gradient(loss_value, im_model.trainable_weights)

			optimizer.apply_gradients(zip(grads, im_model.trainable_weights))
			
			
	print(f"finished training model {i+1}")



In [ ]:
# Load imitative models for inference

im_models = []
for i in range(num_subsets):
	im_model = get_classifier()
	im_model.load_weights(f'./imitative_models/im_model_{i+1}.weights.h5')
	im_models.append(im_model)

In [ ]:
# Calculate scaled confidence score for a query

def scaled_confidence_score(x, y, model):
	pred = model.predict(x, verbose=False)[0]

	true_class = y[0]

	copy = pred.copy()
	copy[true_class] = float('-inf')
	other_max_class = np.argmax(copy)

	probas = tf.nn.softmax(pred)
	true_conf = probas[true_class] + 1e-45
	other_max_conf = probas[other_max_class] + 1e-45
	phi = np.log10(true_conf) - np.log10(other_max_conf)

	return phi

In [ ]:
correct = 0
count = 0

# Run attack on each query in the original dataset

for step, (x, y) in enumerate(ds):
	target_phi = scaled_confidence_score(x, y, target_model)

	in_indexes = []

	# Create list of indexes of subsets that this query was included in
	for offset in range(int(num_subsets / 2)):
		index = (step + offset) % 10
		in_indexes.append(index)

	mean_in_phi = 0
	for i in range(len(im_models)):
		if i in in_indexes:
			mean_in_phi = mean_in_phi + scaled_confidence_score(x, y, im_models[i])
	mean_in_phi = mean_in_phi / (len(im_models)/2)

	mean_out_phi = 0
	for i in range(len(im_models)):
		if i not in in_indexes:
			mean_out_phi = mean_out_phi + scaled_confidence_score(x, y, im_models[i])
	mean_out_phi = mean_out_phi / (len(im_models)/2)

	out_dist = (target_phi - mean_out_phi)**2
	in_dist = (target_phi - mean_in_phi)**2

	
	membership_guess = 0
	if (in_dist < out_dist):
		membership_guess = 1

	if filter[step] == membership_guess:
		correct += 1
	count += 1

advantage = correct / count

print(f'Advantage is {advantage}')